# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Load Data
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv'
df = pd.read_csv(url)

# 2. Basic Cleaning
# Dropping columns with too many missing values or irrelevant info for simple modeling
df = df.drop(columns=['company', 'agent', 'reservation_status_date'])
df = df.dropna()

# 3. Encoding Categorical Variables
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

# 4. Define X and y
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# 5. Split Data (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())

Rows: 9138, Columns: 29


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status
0,0,0,342,2015,5,27,1,0,0,2,...,2,2,3,0,0,2,0.0,0,0,1
1,0,0,737,2015,5,27,1,0,0,2,...,2,2,4,0,0,2,0.0,0,0,1
2,0,0,7,2015,5,27,1,0,1,1,...,0,2,0,0,0,2,75.0,0,0,1
3,0,0,13,2015,5,27,1,0,1,1,...,0,0,0,0,0,2,75.0,0,0,1
4,0,0,14,2015,5,27,1,0,2,2,...,0,0,0,0,0,2,98.0,0,1,1


### ✍️ Your Response:
1. **Rows and Columns:** The dataset contains 9,138 rows and 29 columns.
2. **Feature Types:** It includes categorical features (hotel type, month, deposit type) and numerical features (lead time, number of adults, ADR).
3. **Preparation Steps:** I removed columns with high missing values (`company`, `agent`), dropped remaining rows with null values, and used Label Encoding to convert categorical text data into numeric format for model compatibility.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [2]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Train Naïve Bayes
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict
y_pred_nb = nb_model.predict(X_test)
nb_acc = accuracy_score(y_test, y_pred_nb)

print("--- Naïve Bayes Classification Report ---")
print(classification_report(y_test, y_pred_nb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

--- Naïve Bayes Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2035
           1       0.99      1.00      1.00       707

    accuracy                           1.00      2742
   macro avg       1.00      1.00      1.00      2742
weighted avg       1.00      1.00      1.00      2742

Confusion Matrix:
[[2031    4]
 [   0  707]]


### Reflection:
1. **Performance:** The model performed great with an accuracy of 99.85%. Accuracy and F1-score are both excellent metrics here as the classes are relatively balanced
2. **Utility:** This model is efficient and fast, making it ideal for real-time booking systems to provide immediate risk scores as soon as a customer enters their details.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [3]:
from sklearn.svm import SVC

# Scaling data (Required for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVM (Linear kernel)
# Note: Using a smaller subset if rows > 10k to ensure it finishes within reasonable time
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_scaled, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, y_pred_svm)

print("--- SVM Classification Report ---")
print(classification_report(y_test, y_pred_svm))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

--- SVM Classification Report ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      2035
           1       1.00      0.98      0.99       707

    accuracy                           0.99      2742
   macro avg       0.99      0.99      0.99      2742
weighted avg       0.99      0.99      0.99      2742

Confusion Matrix:
[[2032    3]
 [  13  694]]


### Reflection:
1. **Performance:** SVM achieved 99.42% accuracy. While slightly lower than Naïve Bayes, it is highly accurate.
2. **Business Situations:** SVM is excellent for datasets where the relationship between features isn't simple. It could provide deeper insights into customer segments that have complex cancellations.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [4]:
from sklearn.neural_network import MLPClassifier

# Train Neural Network
# 2 hidden layers with 10 neurons each
mlp_model = MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=500, random_state=42)
mlp_model.fit(X_train_scaled, y_train)

# Predict
y_pred_mlp = mlp_model.predict(X_test_scaled)
mlp_acc = accuracy_score(y_test, y_pred_mlp)

print("--- Neural Network Classification Report ---")
print(classification_report(y_test, y_pred_mlp))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

--- Neural Network Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2035
           1       1.00      1.00      1.00       707

    accuracy                           1.00      2742
   macro avg       1.00      1.00      1.00      2742
weighted avg       1.00      1.00      1.00      2742

Confusion Matrix:
[[2034    1]
 [   2  705]]


### Reflection:
1. **Comparison:** This model was the best with 99.89% accuracy. It captured nearly all cancellations correctly with only 3 total errors.
2. **Black Box Concern:** Businesses might be cautious because it's harder to explain why a specific booking was flagged. Although, the near-perfect accuracy, the performance likely outweighs the lack of interpretability.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [5]:
print(f"Naïve Bayes Accuracy:  {nb_acc:.4f}")
print(f"SVM Accuracy:          {svm_acc:.4f}")
print(f"Neural Network Acc:    {mlp_acc:.4f}")

models = {'Naïve Bayes': nb_acc, 'SVM': svm_acc, 'Neural Network': mlp_acc}
best_model = max(models, key=models.get)
print(f"\nThe best performing model based on accuracy is: {best_model}")

Naïve Bayes Accuracy:  0.9985
SVM Accuracy:          0.9942
Neural Network Acc:    0.9989

The best performing model based on accuracy is: Neural Network


### Reflection:
1. **Comparison:**
   - **Accuracy:** Neural Network (99.89%) > Naïve Bayes (99.85%) > SVM (99.42%).
   - **Ease of Use:** Naïve Bayes was the simplest to implement.
   - **Training Time:** Naïve Bayes was the fastest; SVM and NN required more resources.
2. **Recommendation:** I recommend the Neural Network for its higher predictive power, assuming the hotel has the infrastructure to support it.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response:
1. **Recommendation:** I recommend deploying the **Neural Network** model. It achieved a 99.89% accuracy rate. This helps solve the business problem of unpredictable staffing and revenue loss by allowing management to overbook or adjust rates confidently. A risk is the 'black box' nature, but the high precision minimizes the risk of upsetting guests with incorrect flags. Future data on local events or weather could increase accuracy.
2. **Learning Outcome:** This exercise allows me to use different models to test their predictive accuracy which is what I will be doing with marketing data.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [6]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_12_bayes_svm_neural.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=